Step 1: Loading and Preparing the COCO Data

In [4]:
import json
import os

# Define the path to the data folder relative to the current notebook location
data_dir = os.path.join(os.getcwd(), 'data')

# Load the COCO formatted annotations
coco_path = os.path.join(data_dir, 'result.json')
with open(coco_path) as f:
    cocodata = json.load(f)

# Initialize an empty list to store the data in Huggingface format
huggingdata = []

# Convert the COCO annotations to Huggingface format
for image in cocodata['images']:
    image['file_name'] = os.path.join(data_dir, image['file_name'].split(os.path.sep)[-1])
    image['image_id'] = image['id']
    image['objects'] = {'bbox': [], 'category': [], 'area': [], 'id': []}
    
    # Match annotations to the images
    for annot in cocodata['annotations']:
        if annot['image_id'] == image['id']:
            image['objects']['bbox'].append(annot['bbox'])
            image['objects']['category'].append(annot['category_id'])
            image['objects']['area'].append(annot['area'])
            image['objects']['id'].append(annot['id'])
    
    huggingdata.append(image)

# Write the Huggingface formatted data to a jsonl file
metadata_path = os.path.join(data_dir, "metadata.jsonl")
with open(metadata_path, 'w') as f:
    for item in huggingdata:
        f.write(json.dumps(item) + "\n")

Step 2: Load the Data from the data Folder into a DatasetDict

In [5]:
from datasets import load_dataset

# Load the dataset from the "data" folder using the os path to handle platform differences
candy_data = load_dataset('imagefolder', data_dir=data_dir)

C:\Users\Numan\AppData\Roaming\Python\Python39\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Generating train split: 0 examples [00:00, ? examples/s]


DatasetGenerationError: An error occurred while generating the dataset

Step 3: Ensure the Image Paths Are Correctly Referenced in the candy_counter Function

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

# Define a test image path, assuming it is stored in the "data" folder
test_image_path = os.path.join(data_dir, 'your_test_image.jpg')

# Load and display the test image
image = Image.open(test_image_path)
plt.imshow(image)
plt.show()

# Run your candy_counter function
result = candy_counter(image)
print(result)

Step 4: Preparing the Model for Fine-Tuning

In [ ]:
from transformers import DetrForObjectDetection, DetrFeatureExtractor
from transformers import TrainingArguments, Trainer

# Load pre-trained model and feature extractor
model = DetrForObjectDetection.from_pretrained("facebook/detr-resnet-50")
feature_extractor = DetrFeatureExtractor.from_pretrained("facebook/detr-resnet-50")

# Define the training arguments
training_args = TrainingArguments(
    output_dir='./results',
    per_device_train_batch_size=2,
    num_train_epochs=10,
    logging_steps=10,
    save_steps=10,
    save_total_limit=2,
    evaluation_strategy="epoch"
)

Step 5: Defining Data Collator and Trainer

In [ ]:
import torch

# Define a function to collate the batch (required for PyTorch's DataLoader)
def collate_fn(batch):
    pixel_values = torch.stack([item["pixel_values"] for item in batch])
    labels = [{k: torch.tensor(v) for k, v in t.items()} for t in batch]
    return {"pixel_values": pixel_values, "labels": labels}

# Initialize the Trainer with the model, data, and arguments
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=candy_data["train"],
    eval_dataset=candy_data["test"],
    data_collator=collate_fn
)

Step 6: Train and Save Model

In [ ]:
# Start the training process
trainer.train()

# Save the fine-tuned model
trainer.save_model("candy_detector")

Step 7: Creating the Inference Function: candy_counter

In [ ]:
from transformers import pipeline

# Load the fine-tuned model
obj_detector = pipeline("object-detection", model="candy_detector")

def candy_counter(image):
    detections = obj_detector(image, threshold=0.5)
    
    # Initialize candy type counts
    candy_counts = {
        'Moon': 0, 'Insect': 0, 'Black_star': 0, 'Grey_star': 0,
        'Unicorn_whole': 0, 'Unicorn_head': 0, 'Owl': 0, 'Cat': 0
    }
    
    # Mapping from category IDs to candy labels
    id2label = {1: 'Moon', 2: 'Insect', 3: 'Black_star', 4: 'Grey_star',
                5: 'Unicorn_whole', 6: 'Unicorn_head', 7: 'Owl', 8: 'Cat'}
    
    # Count detected objects
    for detection in detections:
        label_id = detection['label']
        label_name = id2label.get(label_id, None)
        if label_name:
            candy_counts[label_name] += 1
    
    return candy_counts

Step 8: Testing the Model on a Sample Image

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

# Define a test image path (assuming the image is stored in the "data" folder)
test_image_path = os.path.join(data_dir, 'your_test_image.jpg')

# Load and display the test image
image = Image.open(test_image_path)
plt.imshow(image)
plt.show()

# Run your candy_counter function on the test image
result = candy_counter(image)
print(result)